# Running the SE LLM-RE pipeline on Colab

Generates a traceable requirement set for e-prescription issuance, selects an SDLC
model under three prompt framings, and audits the output against ISO/IEC/IEEE 29148.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. On CPU the Part 2
matrix takes well over an hour; on a T4 the whole pipeline is roughly 25-35 minutes.

Run the cells in order. Steps 1-3 are setup, 4-9 are the pipeline.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Expect a T4 with ~15-16 GB. If this errors the runtime is CPU-only - switch it
# under Runtime > Change runtime type before going any further.

## Step 1 - Install and start Ollama

Colab has no systemd, so the daemon is launched by hand and we poll until it answers.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for attempt in range(30):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2).raise_for_status()
        print(f'Ollama is up (after {attempt + 1}s)')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start - re-run this cell')

## Step 2 - Pull both models

~10 GB total. Both are required: the second exists so Part 2 can measure cross-model
agreement, which a single model cannot provide.

**This is the slowest setup step (5-10 min) and Colab does not persist it** - a fresh
runtime means pulling again. See Troubleshooting for a Drive-caching option.

In [ ]:
!ollama pull qwen2.5:7b-instruct
!ollama pull llama3.1:8b
!ollama list

## Step 3 - Clone the repo and install dependencies

Colab already ships torch, pandas, numpy and matplotlib. `requirements.txt` pins no
torch version, so pip keeps the preinstalled GPU build rather than replacing it.

In [ ]:
!git clone -q https://github.com/somyaknotfound/se-llm-requirement.git
%cd /content/se-llm-requirement
!pip install -q -r requirements.txt
print('\ndependencies installed')

## Step 4 - Smoke-test the transport

Do this before anything else; nothing downstream works if the transport is flaky.
Expect a JSON round-trip from both models with latency and token counts.

In [ ]:
!python -m src.llm --smoke

## Step 5 - Build the retrieval index

The corpus is committed, so ingestion is already done. Only the FAISS index needs
rebuilding - it is gitignored as a derived artifact.

**Read the sanity output.** The three probe queries should surface 21 CFR 1311.140
(signing controlled substances), 45 CFR 170.315(a) (CPOE - medications) and the FHIR
audit-logging section. If they do not, retrieval is miscalibrated and everything
downstream inherits the fault.

In [ ]:
!python -m src.index build
!python -m src.index sanity

## Step 6 - Part 1: generate the requirement set

Writes `outputs/requirements.csv`, `requirements_reasoning.csv` and
`traceability_matrix.csv`, plus the raw response for the appendix.

The driver enforces the output contract and makes **one** repair attempt before
aborting. If it aborts, that is itself a reportable result about a 7B model's schema
compliance - read `outputs/raw_p1_repair_response.txt` rather than loosening the
contract until it passes.

In [ ]:
!python -m src.generate_reqs

## Step 7 - Part 2: the SDLC matrix

3 framings x 3 trials x 2 models = 18 runs. **This is the long cell (~10-20 min on a
T4).** Keep the tab alive or Colab may disconnect you mid-run.

Watch the recommendation printed per run. If it tracks the priming rather than the
requirement set, that is the framing-sensitivity finding the report is after.

In [ ]:
!python -m src.select_sdlc

## Step 8 - Validation

Rule-based 29148 scorer, LLM-as-critic (one call per requirement on a fresh context),
and the hallucination audit over every regulatory citation the model produced.

In [ ]:
!python -m src.validate all

## Step 9 - Metrics and figures

In [ ]:
!python -m src.metrics
!python -m src.report

In [ ]:
import pathlib
from IPython.display import Image, display

for png in sorted(pathlib.Path('figures').glob('*.png')):
    print(png.name)
    display(Image(str(png)))

## Step 10 - Get the results off Colab

**Everything under `/content` is destroyed when the runtime ends.** Download first.

In [ ]:
from google.colab import files

!cd /content/se-llm-requirement && zip -qr /content/se_llm_results.zip \
    outputs figures logs report
print('outputs, figures, logs and report zipped')
files.download('/content/se_llm_results.zip')

### Or push the results back to GitHub

Put a GitHub personal access token in **Colab Secrets** (key icon in the left
sidebar, name it `GH_TOKEN`, enable it for this notebook), then run the cell below.

Do not paste a token into a notebook cell - it gets saved into the `.ipynb` and then
into git history.

In [ ]:
from google.colab import userdata

token = userdata.get('GH_TOKEN')

!git config user.name  'somyaknotfound'
!git config user.email 'somyaknotfound@users.noreply.github.com'
!git add -A outputs figures logs report
!git commit -q -m 'Add pipeline results from Colab run'
!git push -q https://{token}@github.com/somyaknotfound/se-llm-requirement.git main
print('pushed')

---

## Troubleshooting

**`cannot reach Ollama at http://127.0.0.1:11434`** - the daemon died with the cell
that started it, or the runtime restarted. Re-run Step 1.

**Slow runs, and `nvidia-smi` shows low GPU use** - both models resident at 16k
context can exceed a T4's 16 GB, and Ollama then falls back to CPU without saying so.
Free VRAM between parts with:

```
!curl -s http://127.0.0.1:11434/api/generate -d '{"model": "qwen2.5:7b-instruct", "keep_alive": 0}'
```

Lowering `num_ctx` in `config/models.yaml` also works, but that invalidates
`outputs/` - rerun from Step 6 rather than mixing results across settings.

**Disconnected mid-matrix** - Part 2 writes its CSVs only at the end, so a disconnect
loses the matrix. Individual calls survive in `logs/llm_calls.jsonl` if the runtime is
still alive; otherwise re-run Step 7.

**Re-pulling 10 GB every session** - mount Drive and point Ollama at it *before*
Step 1:

```python
from google.colab import drive; drive.mount('/content/drive')
import os; os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
```

The first run costs the same; later runs skip the download.

**Still to do by hand afterwards** - adjudicate ~10 conflicts in
`outputs/adjudication_worksheet.csv`, copy the verdicts into the
`human_adjudication` column of `validation_29148.csv`, and fill the `[PENDING RUN]`
sections of `report/report.md`.